In [30]:
import pandas as pd
import glob
import os


datasets = ['restaurant', 'tax', 'hospital', 'sisu']

In [ ]:

# Read all classification CSV files
csv_files = glob.glob('classification_csv_results/llm_classification_*.csv')

# Load all dataframes
dfs = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    dfs.append(df)

# Combine all dataframes
df_combined = pd.concat(dfs, ignore_index=True)

print(f"Total rows: {len(df_combined)}")
print(f"Datasets: {df_combined['dataset'].unique()}")
print(f"Models: {df_combined['model'].unique()}")
print(f"Prompt keys: {df_combined['prompt_key'].unique()}")
print()

def generate_latex_table_for_dataset(df, dataset_name):
    """
    Generate a LaTeX booktabs table for a specific dataset.
    No dataset column since each table is dataset-specific.
    """
    # Filter for specific dataset and sort
    df = df[df['dataset'] == dataset_name].copy()
    df = df.sort_values(['model', 'prompt_key', 'use_context']).reset_index(drop=True)

    # Simplify model names for display
    model_display_names = {
        'gpt-5-mini': 'GPT-5-mini',
        'gpt-5': 'GPT-5',
        'sabiazinho-4': 'Sabiazinho-4',
        'sabia-4': 'Sabia-4',
        'mistral-small-2506': 'Mistral-Small',
        'mistral-large-2512': 'Mistral-Large',
        'llama-3.3-70b-versatile': 'Llama-3.3-70b'
    }

    # Prompt key -> "Use Examples?" (all include the DQC definition)
    examples_display_names = {
        'zero_shot': 'No',
        'zero_shot_with_table': 'No',
        'with_examples': 'Yes',
        'with_examples_and_table': 'Yes',
    
        'zero_shot_pt': 'No',
        'zero_shot_with_table_pt': 'No',
        'with_examples_pt': 'Yes',
        'with_examples_and_table_pt': 'Yes'
    }

    caption = (
        f"Sentence-level rule classification on the {dataset_name.capitalize()} dataset. "
        f"All variants include a data quality constraint definition; ``Examples'' denotes whether generic data-quality examples were added, "
        f"and ``Context'' denotes whether table schema and sample values were provided."
    )


    latex = []
    latex.append(r'\begin{table}[htbp]')
    latex.append(r'\centering')
    latex.append(r'\small')
    latex.append(f'\\caption{{{caption}}}')
    latex.append(f'\\label{{tab:classification_results_{dataset_name}}}')
    latex.append(r'\begin{tabular}{@{}lccccc@{}}')
    latex.append(r'\toprule')
    latex.append(r'\textbf{Model} & \textbf{Examples} & \textbf{Context} & \textbf{Prec.} & \textbf{Rec.} & \textbf{F1} \\')
    latex.append(r'\midrule')

    prev_model = None
    model_rows = {}

    # First pass: count rows per model
    for _, row in df.iterrows():
        model = row['model']
        model_rows[model] = model_rows.get(model, 0) + 1

    # Second pass: generate table rows
    model_first_row = {}

    for _, row in df.iterrows():
        model = row['model']
        prompt_key = row['prompt_key']
        use_examples = examples_display_names.get(prompt_key, prompt_key)  # fallback to raw key if unexpected
        use_context = 'Yes' if bool(row['use_context']) else 'No'
        precision = f"{row['precision']:.2f}"
        recall = f"{row['recall']:.2f}"
        f1 = f"{row['f1_score']:.2f}"

        model_name = model_display_names.get(model, model)

        # Light rule between models (but not at the start)
        if prev_model is not None and model != prev_model:
            latex.append(r'\cmidrule(lr){1-6}')

        # Model multirow
        if model not in model_first_row:
            model_cell = f"\\multirow{{{model_rows[model]}}}{{*}}{{{model_name}}}"
            model_first_row[model] = True
        else:
            model_cell = ""

        latex.append(f"{model_cell} & {use_examples} & {use_context} & {precision} & {recall} & {f1} \\\\")
        prev_model = model

    latex.append(r'\bottomrule')
    latex.append(r'\end{tabular}')
    latex.append(r'\end{table}')

    return '\n'.join(latex)

# Generate one table per dataset
datasets = df_combined['dataset'].unique()
print("Generating tables for each dataset...")
print("="*80)

for dataset in sorted(datasets):
    latex_table = generate_latex_table_for_dataset(df_combined, dataset)

    # Save to file with dataset-specific name
    output_file = f'classification_results_{dataset}.tex'
    with open(output_file, 'w') as f:
        f.write(latex_table)

    print(f"\n{dataset.upper()} DATASET:")
    print(latex_table)
    print(f"\nSaved to: {output_file}")
    print("="*80)

print("\nAll tables generated successfully!")
print("Required package: \\usepackage{booktabs, multirow}")



Total rows: 108
Datasets: <ArrowStringArray>
['sisu', 'hospital', 'restaurant', 'tax', 'person']
Length: 5, dtype: str
Models: <ArrowStringArray>
[     'mistral-small-2506',              'gpt-5-mini',
                   'gpt-5',      'mistral-large-2512',
 'llama-3.3-70b-versatile',            'sabiazinho-4',
                 'sabia-4']
Length: 7, dtype: str
Prompt keys: <ArrowStringArray>
[                 'zero_shot',       'zero_shot_with_table',
              'with_examples',    'with_examples_and_table',
               'zero_shot_pt',    'zero_shot_with_table_pt',
           'with_examples_pt', 'with_examples_and_table_pt']
Length: 8, dtype: str

Generating tables for each dataset...

HOSPITAL DATASET:
\begin{table}[htbp]
\centering
\small
\caption{Sentence-level rule classification on the Hospital dataset. All variants include a data quality constraint definition; ``Examples'' denotes whether generic data-quality examples were added, and ``Context'' denotes whether table schema a

In [32]:
latex_table

"\\begin{table}[htbp]\n\\centering\n\\small\n\\caption{Sentence-level rule classification on the Tax dataset. All variants include a data quality constraint definition; ``Examples'' denotes whether generic data-quality examples were added, and ``Context'' denotes whether table schema and sample values were provided.}\n\\label{tab:classification_results_tax}\n\\begin{tabular}{@{}lccccc@{}}\n\\toprule\n\\textbf{Model} & \\textbf{Examples} & \\textbf{Context} & \\textbf{Prec.} & \\textbf{Rec.} & \\textbf{F1} \\\\\n\\midrule\n\\multirow{4}{*}{GPT-5} & Yes & No & 0.46 & 1.00 & 0.63 \\\\\n & Yes & Yes & 0.50 & 1.00 & 0.67 \\\\\n & No & No & 0.38 & 1.00 & 0.55 \\\\\n & No & Yes & 0.67 & 1.00 & 0.80 \\\\\n\\cmidrule(lr){1-6}\n\\multirow{4}{*}{GPT-5-mini} & Yes & No & 0.30 & 1.00 & 0.46 \\\\\n & Yes & Yes & 0.32 & 1.00 & 0.48 \\\\\n & No & No & 0.26 & 1.00 & 0.41 \\\\\n & No & Yes & 0.29 & 1.00 & 0.44 \\\\\n\\cmidrule(lr){1-6}\n\\multirow{4}{*}{Llama-3.3-70b} & Yes & No & 0.32 & 1.00 & 0.48 \\\